# Transformer Encoder


## 一、准备操作

In [ ]:
# 导包
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
"""
序列建模超参数, 考虑source sentence和target sentence
"""
# 批次大小
batch_size = 2

# 源序列+目标序列词表大小
max_num_src_words = 8
max_num_tgt_words = 8

# 源序列+目标序列序列最大长度
max_src_seq_len = 5
max_tgt_seq_len = 5

In [3]:
"""
源序列+目标序列批次大小为2
源序列: 第一个序列长度为2, 第二个序列长度为4
目标序列: 第一个序列长度为4, 第二个序列长度为3
"""
src_len = torch.Tensor([2, 4]).to(torch.int32)
tgt_len = torch.Tensor([4, 3]).to(torch.int32)

"""
先生成源序列+目标序列
默认用0填充
"""
src_seq = [F.pad(torch.randint(low=1, high=max_num_src_words, size=(L,)), pad=(0, max_src_seq_len-L)) for L in src_len]
tgt_seq = [F.pad(torch.randint(low=1, high=max_num_tgt_words, size=(L,)), pad=(0, max_tgt_seq_len-L)) for L in tgt_len]

"""
再堆叠为批次数据
stack可以避免先unsqueeze再cat
"""
src_seq = torch.stack(src_seq, dim=0)
tgt_seq = torch.stack(tgt_seq, dim=0)

print(f"src_seq:\n{src_seq}")
print(f"tgt_seq:\n{tgt_seq}")

src_seq:
tensor([[6, 1, 0, 0, 0],
        [2, 4, 5, 6, 0]])
tgt_seq:
tensor([[3, 7, 4, 6, 0],
        [5, 2, 6, 0, 0]])


In [4]:
"""
构建嵌入向量表, 将单词序号转化为嵌入向量
形状为(max_num_words+1, model_dim)
+1是为了给第0个pad留位置
"""
model_dim = 8
src_embedding_table = nn.Embedding(max_num_src_words+1, model_dim)
tgt_embedding_table = nn.Embedding(max_num_tgt_words+1, model_dim)

print(f"src_embedding_table:\n{src_embedding_table.weight}")
print(f"src_seq:\n{src_seq}")
print("seq中的一个序号, 表示嵌入表中的对应的行, 如0表示embedding中的第0行向量")
print(f"src_embedding:\n{src_embedding_table(src_seq)}")

src_embedding_table:
Parameter containing:
tensor([[-1.7780e-01,  1.1419e+00,  8.5016e-01, -2.0585e+00, -4.8233e-01,
         -7.8656e-01,  1.0506e+00,  1.0233e+00],
        [-1.0912e+00,  4.4798e-01, -7.8677e-01,  4.2916e-02,  6.1478e-01,
          5.0873e-01, -5.4889e-01,  8.7532e-01],
        [-3.1162e-01,  9.7491e-01,  1.4431e+00,  1.3338e-01, -1.1168e+00,
         -1.6761e+00, -1.4480e+00, -9.3380e-02],
        [ 1.6420e+00, -1.7893e+00,  1.9226e+00, -5.8849e-02, -1.5688e-01,
         -1.2090e+00, -3.9921e-01, -1.0281e+00],
        [-5.7250e-01, -1.1698e+00,  1.0480e+00, -2.6602e-01,  6.2869e-01,
         -2.2809e+00,  5.6216e-01, -1.6845e-01],
        [-4.3934e-01,  6.1937e-02,  5.3666e-01,  8.8272e-02,  7.7732e-01,
          1.7052e+00, -1.3942e-01,  1.3230e+00],
        [-2.0208e-01, -1.4249e-03,  2.0027e-01,  1.6514e-01,  1.0587e+00,
          6.2196e-01,  5.7122e-01, -4.5505e-01],
        [ 9.6746e-01, -3.0924e-02, -1.4216e-01, -6.7744e-02,  2.4138e-01,
          1.6636e-01, 

## 二、位置嵌入

In [5]:
"""
最大pos即为序列最长长度
"""
max_position_len = max_src_seq_len

"""
构建pos_mat和i2_mat, 计算三角函数内的值
pos_mat形状为(max_position_len, 1)
i2_mat形状为(1, model_dim/2)
    假设2*i=0,2,4,6
    则pe中的偶数列(2i)的嵌入值为2*i
    则pe中的奇数列(2i+1)也嵌入值为2*i
    另一种求法参考**position_embedding节**
"""
pos_mat = torch.arange(max_position_len).reshape((-1, 1))
i2_mat = torch.pow(10000, torch.arange(0, model_dim, 2).reshape((1, -1)) / model_dim)

"""
初始化位置嵌入矩阵, 形状为(max_position_len, model_dim)
位置嵌入维度和词表嵌入维度相等
"""
position_embedding = torch.zeros((max_position_len, model_dim))
position_embedding[:, 0::2] = torch.sin(pos_mat / i2_mat)
position_embedding[:, 1::2] = torch.cos(pos_mat / i2_mat)

"""
利用pytorch的Embedding接口, 构建位置嵌入矩阵
直接复制给position_embedding_table.weight即可
"""
position_embedding_table = nn.Embedding(max_position_len, model_dim)
position_embedding_table.weight = nn.Parameter(position_embedding, requires_grad=False)

# 构造源序列位置: 对于批次中的一个词, [0, ..., max_pos-1]
src_pos = torch.stack([torch.arange(max_src_seq_len) for _ in src_len], dim=0).to(torch.int32)
print(f"src_pos:\n{src_pos}")
print(f"src_pe:\n{position_embedding_table(src_pos)}")

src_pos:
tensor([[0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]], dtype=torch.int32)
src_pe:
tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00],
         [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
           9.9920e-01,  4.0000e-03,  9.9999e-01]],

        [[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01, 